# Entrena a tu propio David

Ajusta **Gemma 4 E2B** con LoRA para que convierta mensajes de WhatsApp de una ferretería en tickets JSON.
Sigue el codelab paso a paso: cada sección de este notebook corresponde a un paso.

- Entorno: Colab con GPU **T4** (menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*).
- No uses *Ejecutar todo*: corre las celdas en orden, una por una, mientras lees el paso.
- Código y datos: [https://github.com/ykro/codelabs-2027/tree/main/entrena-tu-david](https://github.com/ykro/codelabs-2027/tree/main/entrena-tu-david)

## Paso 3 · Abre el notebook e instala Unsloth

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install --no-deps unsloth==2026.10.4 unsloth_zoo==2026.10.4 bitsandbytes==0.50.2 trl==1.15.0 cut_cross_entropy==25.1.1
!pip install sentencepiece protobuf "datasets==4.8.5" "huggingface_hub>=1.5.0,<2.0" hf_transfer gradio==6.26.0
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0" "torchao>=0.16.0"

In [ ]:
import unsloth, transformers, trl, peft, gradio
print("unsloth", unsloth.__version__, "· transformers", transformers.__version__,
      "· trl", trl.__version__, "· peft", peft.__version__, "· gradio", gradio.__version__)

## Paso 4 · Carga Gemma 4 E2B

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=1024,
    load_in_4bit=False,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

In [ ]:
import torch

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} · memoria total: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Memoria reservada después de cargar: {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB")

## Paso 5 · Mira el dataset y escribe el prompt

In [ ]:
import json, os, urllib.request

URL = "https://raw.githubusercontent.com/ykro/codelabs-2027/main/entrena-tu-david/datos"
for nombre in ["train.jsonl", "test.jsonl"]:
    if not os.path.exists(nombre):  # si subiste el archivo a Colab, usa el tuyo
        urllib.request.urlretrieve(f"{URL}/{nombre}", nombre)

train = [json.loads(linea) for linea in open("train.jsonl", encoding="utf-8")]
test = [json.loads(linea) for linea in open("test.jsonl", encoding="utf-8")]
print(f"train: {len(train)} ejemplos · test: {len(test)} ejemplos")

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.DataFrame([{"mensaje": r["mensaje"], **r["ticket"]} for r in train[:5]], dtype=object)

In [ ]:
print(json.dumps(train[0]["origen"], ensure_ascii=False, indent=1))

In [ ]:
# TODO: Escribe aquí PROMPT y la función conversacion() (paso 5 del codelab).


## Paso 6 · Mide el modelo base

In [ ]:
import re, unicodedata

CAMPOS = ["tipo", "categoria", "producto", "cantidad", "monto_q", "urgencia"]
PERMITIDOS = {
    "tipo": ["pedido", "cotizacion", "reclamo", "seguimiento"],
    "categoria": ["obra_gris", "techos", "plomeria", "electricidad", "pintura", "herramientas", "jardin"],
    "urgencia": ["alta", "media", "baja"],
}


def normalizar(texto):
    t = unicodedata.normalize("NFKD", str(texto).lower())
    return " ".join("".join(c for c in t if not unicodedata.combining(c)).split())


def leer_ticket(respuesta):
    """El ticket que escribió el modelo, o None si no es un JSON válido con los 6 campos."""
    texto = re.sub(r"^```(?:json)?|```$", "", respuesta.strip()).strip()
    try:
        ticket = json.loads(texto)
    except json.JSONDecodeError:
        return None
    if not isinstance(ticket, dict) or set(ticket) != set(CAMPOS):
        return None
    if any(ticket[c] not in valores for c, valores in PERMITIDOS.items()):
        return None
    return ticket


def igual(campo, a, b):
    if campo in ("cantidad", "monto_q"):
        try:
            return (a is None and b is None) or abs(float(a) - float(b)) < 0.01
        except (TypeError, ValueError):
            return False
    return normalizar(a) == normalizar(b)


def calificar(respuestas, filas):
    validos, perfectos, aciertos = 0, 0, {c: 0 for c in CAMPOS}
    for respuesta, fila in zip(respuestas, filas):
        ticket = leer_ticket(respuesta)
        if ticket is None:
            continue
        validos += 1
        bien = [igual(c, ticket[c], fila["ticket"][c]) for c in CAMPOS]
        for c, ok in zip(CAMPOS, bien):
            aciertos[c] += ok
        perfectos += all(bien)
    n = len(filas)
    return {"JSON válido": validos / n,
            "Campos correctos": sum(aciertos.values()) / (n * len(CAMPOS)),
            "Tickets perfectos": perfectos / n,
            **{f"· {c}": aciertos[c] / n for c in CAMPOS}}


def tabla(**columnas):
    return pd.DataFrame(columnas).map(lambda x: f"{x:.0%}")

In [ ]:
import time

texto_tok = getattr(tokenizer, "tokenizer", tokenizer)


def responder(mensajes, lote=16):
    texto_tok.padding_side = "left"  # para generar, el relleno va a la izquierda
    salidas = []
    for i in range(0, len(mensajes), lote):
        prompts = [tokenizer.apply_chat_template(conversacion(m), tokenize=False, add_generation_prompt=True)
                   for m in mensajes[i:i + lote]]
        entrada = texto_tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        salida = model.generate(**entrada, max_new_tokens=96, do_sample=False)
        salidas += texto_tok.batch_decode(salida[:, entrada["input_ids"].shape[1]:], skip_special_tokens=True)
    return salidas

In [ ]:
FastModel.for_inference(model)
inicio = time.time()
respuestas_base = responder([r["mensaje"] for r in test])
print(f"{len(test)} mensajes en {time.time() - inicio:.0f} s")
base = calificar(respuestas_base, test)
tabla(**{"Gemma sin ajustar": base})

In [ ]:
for fila, respuesta in list(zip(test, respuestas_base))[:3]:
    print("MENSAJE  ", fila["mensaje"])
    print("ESPERADO ", json.dumps(fila["ticket"], ensure_ascii=False))
    print("BASE     ", " ".join(respuesta.split()), "\n")

## Paso 7 · Configura LoRA

In [ ]:
# TODO: Configura el adaptador LoRA con FastModel.get_peft_model (paso 7 del codelab).


## Paso 8 · Entrena

In [ ]:
from datasets import Dataset

textos = [tokenizer.apply_chat_template(conversacion(r["mensaje"], r["ticket"]), tokenize=False).removeprefix("<bos>")
          for r in train]
datos = Dataset.from_dict({"text": textos})
print(textos[0])

In [ ]:
# TODO: Configura el entrenamiento con SFTTrainer y SFTConfig (paso 8 del codelab).


In [ ]:
FastModel.for_training(model)
stats = trainer.train()
print(f"Entrenamiento: {stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Pico de memoria de GPU: {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB de {gpu.total_memory / 1024**3:.1f} GB")

## Paso 9 · Mide tu David

In [ ]:
FastModel.for_inference(model)
respuestas_david = responder([r["mensaje"] for r in test])
david = calificar(respuestas_david, test)
tabla(**{"Gemma sin ajustar": base, "Tu David": david})

In [ ]:
cambios = [(f, b, d) for f, b, d in zip(test, respuestas_base, respuestas_david)
           if leer_ticket(d) and all(igual(c, leer_ticket(d)[c], f["ticket"][c]) for c in CAMPOS)
           and not (leer_ticket(b) and all(igual(c, leer_ticket(b)[c], f["ticket"][c]) for c in CAMPOS))]
print(f"{len(cambios)} mensajes que el modelo base fallaba y tu David resuelve. Tres de ellos:\n")
for fila, b, d in cambios[:3]:
    print("MENSAJE ", fila["mensaje"])
    print("BASE    ", " ".join(b.split()))
    print("DAVID   ", d.strip(), "\n")

In [ ]:
model.save_pretrained("david_lora")
_ = tokenizer.save_pretrained("david_lora")
!du -sh david_lora

### Opcional: compáralo con Gemini (Goliat)
Necesitas tu llave de AI Studio guardada en los *Secretos* de Colab con el nombre `GEMINI_API_KEY`. Sin llave, la celda solo avisa y puedes seguir.

In [ ]:
# Opcional: si no tienes llave de AI Studio, esta celda solo avisa y puedes seguir al paso 10.
from google import genai
from google.colab import userdata

MODELO_GEMINI = "gemini-3.8-flash"  # si llegas al límite del día: "gemini-3.5-flash-lite"
try:
    cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
except Exception:
    cliente = None
    print("No encontré GEMINI_API_KEY en los secretos de Colab (o no le diste acceso). Salta este paso.")

if cliente:
    INSTRUCCIONES = (PROMPT.split("\n\nMensaje:")[0] + "\nVas a recibir varios mensajes numerados. "
                     "Responde con un arreglo JSON: un ticket por mensaje, en el mismo orden.")
    respuestas_gemini = []
    for i in range(0, len(test), 25):  # 25 mensajes por solicitud: 6 solicitudes en total
        grupo = test[i:i + 25]
        lista = "\n".join(f"{n}. {r['mensaje']}" for n, r in enumerate(grupo, 1))
        inicio = time.time()
        r = cliente.models.generate_content(model=MODELO_GEMINI, contents=f"{INSTRUCCIONES}\n\nMensajes:\n{lista}",
                                            config={"response_mime_type": "application/json", "temperature": 0})
        tickets = json.loads(r.text)[:len(grupo)]
        respuestas_gemini += [json.dumps(t, ensure_ascii=False) for t in tickets] + [""] * (len(grupo) - len(tickets))
        print(f"{len(respuestas_gemini)}/{len(test)} mensajes ({time.time() - inicio:.0f} s)")
    gemini = calificar(respuestas_gemini, test)
    display(tabla(**{"Gemma sin ajustar": base, "Tu David": david, "Gemini (Goliat)": gemini}))

## Paso 10 · Pruébalo en una app

In [ ]:
# TODO: Escribe la app de Gradio (paso 10 del codelab).


## Paso 11 · Llévalo a tu laptop con Ollama (opcional)
Exporta solo el adaptador a GGUF (unos 50 MB). El modelo completo no cabe en la RAM de Colab gratis al convertirlo.

In [ ]:
!git -c advice.detachedHead=false clone -q --depth 1 --branch b11541 https://github.com/ggml-org/llama.cpp
!pip install -q --no-deps ./llama.cpp/gguf-py

In [ ]:
!python llama.cpp/convert_lora_to_gguf.py david_lora --outfile david-lora.gguf --outtype f16 2>&1 | tail -n 2
!ls -lh david-lora.gguf